# 05 — Modelo Linear Regression

Treina uma regressão linear com `StandardScaler` para prever temperatura do ar (SP).  
Avalia com MAE / RMSE / R² e salva predições e métricas em Parquet para o notebook de comparação (07).

> **Por que StandardScaler?** As features têm escalas muito distintas (`latitude` ≈ -23, `pressao` ≈ 1000, `hora_num` 0–23). Sem normalização, o gradiente do solver diverge e os coeficientes perdem sentido.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import round as spark_round

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

## 1. SparkSession

In [ ]:
spark = (
    SparkSession.builder
    .appName("Linear_Regression_Weather_SP")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 2. Carregamento das bases de treino e teste

In [ ]:
train_path = "/home/jovyan/work/data/processed/weather_sp_train"
test_path  = "/home/jovyan/work/data/processed/weather_sp_test"

df_treino = spark.read.parquet(train_path)
df_teste  = spark.read.parquet(test_path)

print(f"Treino: {df_treino.count():,} linhas")
print(f"Teste : {df_teste.count():,} linhas")

anos_treino = sorted([r["ano"] for r in df_treino.select("ano").distinct().collect()])
anos_teste  = sorted([r["ano"] for r in df_teste.select("ano").distinct().collect()])
print(f"Anos TREINO : {anos_treino}")
print(f"Anos TESTE  : {anos_teste}")

## 3. Definição de features e variável-alvo

In [ ]:
COLUNA_ALVO = "temperatura"

features_candidatas = [
    "ano", "mes", "dia", "hora_num",
    "latitude", "longitude", "height",
    "umidade", "umidade_maxima", "umidade_minima",
    "pressao", "pressao_maxima", "pressao_minima",
    "precipitacao", "radiacao",
    "velocidade_vento", "rajada_vento", "direcao_vento"
]

features_cols = [c for c in features_candidatas if c in df_treino.columns]

print(f"Features utilizadas ({len(features_cols)}):")
for f in features_cols:
    print(f"  - {f}")

## 4. Pipeline: VectorAssembler → StandardScaler → LinearRegression

In [ ]:
# Empacota todas as features em um único vetor
assembler = VectorAssembler(
    inputCols=features_cols,
    outputCol="features",
    handleInvalid="skip"
)

# Normaliza para média=0 e desvio=1 — essencial para a LinearRegression convergir
scaler = StandardScaler(
    inputCol="features",
    outputCol="features_scaled",
    withStd=True,
    withMean=True
)

# Regressão Linear com regularização L2 (Ridge)
lr = LinearRegression(
    featuresCol="features_scaled",
    labelCol=COLUNA_ALVO,
    predictionCol="prediction",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0
)

pipeline_lr = Pipeline(stages=[assembler, scaler, lr])
print("Pipeline definido: assembler → scaler → LinearRegression")

## 5. Treinamento

In [ ]:
print("Treinando Linear Regression...")
modelo_lr = pipeline_lr.fit(df_treino)
print("Treinamento concluído.")

## 6. Predições

In [ ]:
predicoes_lr = modelo_lr.transform(df_teste)

predicoes_lr.select(
    "ano", "mes", "hora_num",
    F.round(COLUNA_ALVO, 2).alias("temperatura_real"),
    F.round("prediction", 2).alias("temperatura_prevista")
).show(20, truncate=False)

## 7. Avaliação: MAE, RMSE, R²

In [ ]:
resultados_lr = {}
for metrica in ["mae", "rmse", "r2"]:
    avaliador = RegressionEvaluator(
        labelCol=COLUNA_ALVO, predictionCol="prediction", metricName=metrica
    )
    resultados_lr[metrica] = avaliador.evaluate(predicoes_lr)

mae_lr  = resultados_lr["mae"]
rmse_lr = resultados_lr["rmse"]
r2_lr   = resultados_lr["r2"]

print("\n" + "="*45)
print("  Métricas — Linear Regression")
print("="*45)
print(f"  MAE  : {mae_lr:.4f} °C")
print(f"  RMSE : {rmse_lr:.4f} °C")
print(f"  R²   : {r2_lr:.4f}")
print("="*45)

## 8. Interpretação dos Coeficientes

Como as features foram normalizadas, a magnitude de cada coeficiente indica a importância relativa da variável.

In [ ]:
lr_treinado = modelo_lr.stages[-1]

coeficientes = list(zip(features_cols, lr_treinado.coefficients.toArray()))
coeficientes_ord = sorted(coeficientes, key=lambda x: abs(x[1]), reverse=True)

print(f"Intercepto: {lr_treinado.intercept:.4f}")
print(f"\nTop 10 features por magnitude do coeficiente:")
print(f"  {'Feature':<25} {'Coeficiente':>12}")
print(f"  {'-'*25} {'-'*12}")
for feat, coef in coeficientes_ord[:10]:
    print(f"  {feat:<25} {coef:>12.4f}")

## 9. Avaliação por Ano

In [ ]:
predicoes_lr.groupBy("ano").agg(
    F.count("*").alias("total_registros"),
    F.round(F.avg(COLUNA_ALVO), 2).alias("temp_media_real"),
    F.round(F.avg("prediction"), 2).alias("temp_media_prevista"),
    F.round(F.avg(F.abs(F.col(COLUNA_ALVO) - F.col("prediction"))), 4).alias("mae_por_ano")
).orderBy("ano").show(truncate=False)

---
## 10. Salvamento dos Resultados

Salva predições, métricas e modelo — necessário para o notebook **07_comparacao_final**.

In [ ]:
# Predições
predicoes_path = "/home/jovyan/work/data/processed/predicoes_linear_regression"

predicoes_lr.select(
    "ano", "mes", "dia", "hora_num", COLUNA_ALVO, "prediction"
).write.mode("overwrite").parquet(predicoes_path)

print(f"Predições salvas em: {predicoes_path}")

In [ ]:
# Métricas — necessário para 07_comparacao_final carregar sem retreinar
df_metricas_lr = spark.createDataFrame(
    [("Linear Regression", float(mae_lr), float(rmse_lr), float(r2_lr))],
    ["modelo", "mae", "rmse", "r2"]
)

metricas_path = "/home/jovyan/work/data/processed/metricas_linear_regression"

df_metricas_lr.write.mode("overwrite").parquet(metricas_path)

print(f"Métricas salvas em: {metricas_path}")
df_metricas_lr.show(truncate=False)

In [ ]:
# Modelo treinado
modelo_path = "/home/jovyan/work/models/linear_regression_weather"

modelo_lr.write().overwrite().save(modelo_path)

print(f"Modelo salvo em: {modelo_path}")

## Conclusão

Resultados salvos para o notebook de comparação (07):
- Predições → `predicoes_linear_regression`
- Métricas  → `metricas_linear_regression`
- Modelo    → `models/linear_regression_weather`